extract features \
↓ \
remove unknown labels \
↓ \
stratified train/test split \
↓ \
train SVM \
↓ \
evaluate classifier \
↓ \
retrain on all labelled cells \
↓ \
predict phenotypes \
↓ \
use probability thresholds to refine annotations \

In [1]:
# import dependencies
%matplotlib inline
import os
import sys
import spatioev as se
import scanpy as sc
import pandas as pd

# Import Scimap
import scimap as sm

from spatioev.config import ClusteringConfig

Running SCIMAP  2.3.5


/Users/shihongwu/anaconda3/envs/spatioev_env/lib/python3.11/site-packages/mpl_scatter_density/__init__.py:4: UserWarning:

pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.

/Users/shihongwu/anaconda3/envs/spatioev_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning:

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html



In [2]:
# set a working directory
wdir = ('/Users/shihongwu/SpatioEv')
os.chdir(wdir)

### Load dataset

In [3]:
adata = se.load_h5ad("data/exp_1.h5ad")
adata

AnnData object with n_obs × n_vars = 51932 × 25
    obs: 'label', 'area', 'eccentricity', 'major_axis_length', 'minor_axis_length', 'perimeter', 'convex_area', 'equivalent_diameter', 'orientation', 'solidity', 'feret_diameter_max', 'Y_centroid', 'X_centroid', 'major_minor_axis_ratio', 'perim_square_over_area', 'major_axis_equiv_diam_ratio', 'convex_hull_resid', 'centroid_dif', 'num_concavities', 'circularity', 'fractal_dimension', 'boundary_irregularity', 'nc_ratio', 'cell_size_nuclear', 'CD11c_nuclear', 'CD14_nuclear', 'CD146_nuclear', 'CD19_nuclear', 'CD3_nuclear', 'CD31_nuclear', 'CD4_nuclear', 'CD45_nuclear', 'CD68_nuclear', 'CD90_nuclear', 'CK19_nuclear', 'COL1A1_nuclear', 'COL4A1_nuclear', 'COL6A1_nuclear', 'FN_nuclear', 'HLADR_nuclear', 'IGFBP5_nuclear', 'NaKATPase_nuclear', 'PCK_nuclear', 'PDPN_nuclear', 'RS6_nuclear', 'SMA_nuclear', 'Tenascin_nuclear', 'Vimentin_nuclear', 'dapi_nuclear', 'label_nuclear', 'area_nuclear', 'eccentricity_nuclear', 'major_axis_length_nuclear', 'min

### Load manual annotation

In [4]:
# 1) Read annotations CSV (saved from adata_final.obs.to_csv, so index is first column)
ann = pd.read_csv("results/phenotyping_annotations.csv", index_col=0)

# 2) Optional: keep only columns you want
ann = ann[["annotated_clusters_update3"]]

# 3) Join into adata.obs by cell ID (index)
adata.obs = adata.obs.join(ann, how="left")

### QC

In [5]:
adata_full = adata.copy()

In [6]:
adata = adata[adata.obs["annotated_clusters_update3"]!="noise"].copy()

### Define marker features

In [7]:
marker_features = [
    "CK19","CD45",
    "CD3","CD4",
    "CD68","CD11c","CD14",
    "Vimentin","COL4A1",
    "SMA","PDPN","CD90"
]

### Define morphology features

In [8]:
morph_features = [
    'area',
    'eccentricity',
    'major_axis_length',
    'minor_axis_length',
    'perimeter',
    'convex_area',
    'equivalent_diameter',
    'solidity',
    'feret_diameter_max',
    'major_minor_axis_ratio',
    'perim_square_over_area',
    'major_axis_equiv_diam_ratio',
    'convex_hull_resid',
    'centroid_dif',
    'num_concavities',
    'circularity',
    'fractal_dimension',
    'boundary_irregularity',
    'nc_ratio'
]

### Build feature matrix

In [9]:
X = se.build_feature_matrix(
    adata,
    markers=marker_features,
    morph_weight=0.4
)

X.shape

(48589, 31)

### Select labelled cells for training

In [10]:
label_key = "annotated_clusters_update3"

train_mask = adata.obs[label_key] != "Unknown"

X_train = X[train_mask]

y_train = adata.obs[label_key][train_mask]

### Train SVM classifier

In [11]:
model, report = se.train_svm_classifier(
    X_train,
    y_train
)

### Evaluate classifier performance

In [12]:
report_df = pd.DataFrame(report).transpose()

report_df

,precision,recall,f1-score,support
CD4 T cells,0.670588,0.888312,0.764246,385.000000
CD90+ CAFs,0.333333,0.804348,0.471338,46.000000
Dendritic cells,0.242991,0.764706,0.368794,34.000000
Macrophages,0.500000,0.909091,0.645161,99.000000
Monocytes,0.494565,0.870813,0.630849,209.000000
PDPN+ CAFs,0.490975,0.800000,0.608501,170.000000
T cells,0.419940,0.920530,0.576763,151.000000
myCAFs,0.899885,0.699463,0.787116,1118.000000
other immune cells,0.751832,0.770386,0.760996,932.000000
tumour,0.994988,0.883856,0.936135,5166.000000


### Predict phenotypes for all cells

In [13]:
pred, prob_df = se.predict_svm(
    model,
    X,
    model.classes_
)

### Store predictions in AnnData

In [14]:
adata.obs["svm_prediction"] = pred

for col in prob_df.columns:
    adata.obs[col] = prob_df[col].values

In [15]:
adata.obs["svm_prediction"].value_counts()

svm_prediction
tumour                23730
myCAFs                 5319
other immune cells     5300
vessel                 3832
CD4 T cells            2574
PDPN+ CAFs             1974
Monocytes              1887
T cells                1784
Macrophages             975
CD90+ CAFs              688
Dendritic cells         526
Name: count, dtype: int64

In [16]:
pd.crosstab(
    adata.obs["annotated_clusters_update3"],
    adata.obs["svm_prediction"]
)

svm_prediction,CD4 T cells,CD90+ CAFs,Dendritic cells,Macrophages,Monocytes,PDPN+ CAFs,T cells,myCAFs,other immune cells,tumour,vessel
annotated_clusters_update3,,,,,,,,,,,
CD4 T cells,1770,4,4,2,8,29,72,8,3,0,26
CD90+ CAFs,1,221,0,1,0,2,1,2,0,2,0
Dendritic cells,0,0,163,2,0,0,0,2,2,1,1
Macrophages,0,11,0,479,1,1,1,0,0,0,0
Monocytes,8,4,3,19,976,15,1,6,1,3,11
PDPN+ CAFs,31,10,5,12,26,716,17,3,28,2,0
T cells,5,3,1,0,1,9,730,2,0,0,4
Unknown,12,134,36,50,104,566,32,981,502,887,577
myCAFs,262,118,239,119,165,131,235,3991,278,33,19


In [17]:
image_path = "background/OnTIMEr18_n_a_backsub.ome.tif"

In [17]:
# Visualize the image with the annotated clusters
sm.pl.image_viewer(image_path=image_path, 
                   adata=adata, 
                   overlay='svm_prediction', 
                   point_size=6,
                   point_color='white')

In [ ]:
se.inspect_reassigned_cells(
    adata,
    image_path=image_path,
    original_value="Unknown"
)

Cells originally 'Unknown': 3881


In [ ]:
se.inspect_disagreements(
    adata,
    image_path=image_path
)

save the results

In [18]:
adata_full.obs["svm_prediction"] = "noise"

adata_full.obs.loc[
    adata.obs.index,
    "svm_prediction"
] = adata.obs["svm_prediction"]

In [19]:
adata_full.obs[
    [
        "annotated_clusters_update3",
        "svm_prediction"
    ]
].to_csv(
    "results/svm_phenotyping_results.csv"
)

In [22]:
import numpy as np

adata_full.obs[prob_df.columns] = np.nan

adata_full.obs.loc[
    adata.obs.index,
    prob_df.columns
] = prob_df.values

In [23]:
prob_cols = [c for c in adata_full.obs.columns if c.startswith("svm_prob_")]

adata_full.obs[prob_cols].to_csv(
    "results/svm_phenotyping_probabilities.csv"
)